### Check Code Data Preprocessing

In [2]:
import pandas as pd

Then /root/MIMICIV/src/mimiciv_clinical_dataset_tabular_death_visit.csv has been created using the tabular.py script. Let's go further and see how the dataset landmark_df.csv has been created.

In [3]:
# Importing libraries (exactly as from xgboost.ipynb)
import ast
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack

In [3]:
# Load dataset
df = pd.read_csv("mimiciv_clinical_dataset_tabular_death_visit_evo.csv")

# Count admission categories
admission_counts = df['admission_category'].value_counts()

# Display counts
print("Admission category counts:")
print(admission_counts := df['admission_category'].value_counts())

Admission category counts:
admission_category
NORMAL       291667
EMERGENCY    254361
Name: count, dtype: int64


In [4]:
df['subject_id'].nunique()

223452

In [5]:
# For debugging and for checking corretness of the code
patients_2_check = df['subject_id'].unique().tolist() # 223452 patients

In [6]:
patients_2_check[0] in patients_2_check

True

In [7]:
ast.literal_eval(df['diagnosis_list'][0])

['Portal hypertension',
 'Other ascites',
 'Cirrhosis of liver without mention of alcohol',
 'Unspecified viral hepatitis C without hepatic coma',
 'Chronic airway obstruction, not elsewhere classified',
 'Bipolar disorder, unspecified',
 'Posttraumatic stress disorder',
 'Personal history of tobacco use']

In [8]:
def safe_join(x):
    if pd.isna(x) or x in ['[]', '', 'None', 'NaN', 'na', 'nan']:
        return ''
    try:
        x_eval = ast.literal_eval(x)
        if isinstance(x_eval, list):
            return '\n'.join(x_eval)
        else:
            return ''
    except:
        return ''

landmark_rows = []

#df = df.sort_values(['subject_id', 'hadm_id']).reset_index(drop=True) # That's not the right ordering!
# the dataset is still ordered, however we should consider a more robust way of doing so.

In [7]:
df.head()

,subject_id,hadm_id,age_at_event,gender,admission_type,admission_category,discharge_location,death_flag,age_at_death,died_during_visit,...,days_until_next_visit,icu_admissions,icu_hours,num_diagnoses,diagnosis_list,num_procedures,procedure_list,num_medications,medication_list,dose_list
0,10000032,22595853,52,F,URGENT,EMERGENCY,HOME,1,52.0,0,...,50.0,0.0,0.0,8.0,"['Portal hypertension', 'Other ascites', 'Cirr...",1.0,['Percutaneous abdominal drainage'],14.0,"['Furosemide', 'Ipratropium Bromide Neb', 'Pot...","['40 mg of Furosemide in TAB through PO/NG', '..."
1,10000032,22841357,52,F,EW EMER.,EMERGENCY,HOME,1,52.0,0,...,25.0,0.0,0.0,8.0,['Unspecified viral hepatitis C with hepatic c...,1.0,['Percutaneous abdominal drainage'],15.0,"['Furosemide', 'Rifaximin', 'Sodium Chloride 0...","['40 mg of Furosemide in TAB through PO/NG', '..."
2,10000032,29079034,52,F,EW EMER.,EMERGENCY,HOME,1,52.0,0,...,11.0,1.0,0.0,13.0,"['Other iatrogenic hypotension', 'Chronic hepa...",0.0,NaN,24.0,"['Bisacodyl', 'Senna', 'Calcium Carbonate', 'R...","['10 mg of Bisacodyl in SUPP through PR', '10 ..."
3,10000032,25742920,52,F,EW EMER.,EMERGENCY,HOSPICE,1,52.0,0,...,-1.0,0.0,0.0,10.0,['Chronic hepatitis C without mention of hepat...,1.0,['Percutaneous abdominal drainage'],28.0,"['Sodium Chloride 0.9% Flush', '0.9% Sodium C...",['3 mL of Sodium Chloride 0.9% Flush in SYR t...
4,10000068,25022803,19,F,EU OBSERVATION,NORMAL,NaN,0,NaN,0,...,-1.0,0.0,0.0,1.0,"['Alcohol abuse, unspecified']",1.0,['Other nonoperative respiratory measurements'],0.0,NaN,NaN


In [ ]:
landmark_rows = []

for patient_id, group in df.groupby('subject_id'):
    group = group.reset_index(drop=True)
    n_visits = group.shape[0]

    # For debugging
    #print(f"Patient {patient_id} in patients_2_check: {patient_id in patients_2_check}")
    patients_2_check.remove(patient_id) if patient_id in patients_2_check else None

    group['medication_list'] = group['medication_list'].apply(safe_join)
    group['diagnosis_list'] = group['diagnosis_list'].apply(safe_join)
    group['procedure_list'] = group['procedure_list'].apply(safe_join)
    group['dose_list'] = group['dose_list'].apply(safe_join)

    for landmark_idx in range(n_visits):
        current_visit = group.iloc[landmark_idx]

        all_visit_so_far = group.iloc[:landmark_idx + 1]

        current_medications = set(current_visit['medication_list'].split('\n'))
        current_diagnoses = set(current_visit['diagnosis_list'].split('\n'))
        current_procedures = set(current_visit['procedure_list'].split('\n'))
        current_dose = set(current_visit['dose_list'].split('\n'))

        if landmark_idx > 0:
            # Get past visits
            past_visits = group.iloc[(landmark_idx-1):landmark_idx]
            past_medications = set("\n".join(past_visits['medication_list']).split('\n'))
            past_diagnoses = set("\n".join(past_visits['diagnosis_list']).split('\n'))
            past_procedures = set("\n".join(past_visits['procedure_list']).split('\n'))
            past_dose = set("\n".join(past_visits['dose_list']).split('\n'))

            new_medications = current_medications.difference(past_medications)
            new_diagnoses = current_diagnoses.difference(past_diagnoses)
            new_procedures = current_procedures.difference(past_procedures)
            new_dose = current_dose.difference(past_dose)
            no_more_diagnoses = past_diagnoses.difference(current_diagnoses)
            no_more_medications = past_medications.difference(current_medications)
            no_more_procedures = past_procedures.difference(current_procedures)
            no_more_dose = past_dose.difference(current_dose)
        else:
            new_medications = current_medications
            new_diagnoses = current_diagnoses
            new_procedures = current_procedures
            new_dose = current_dose
            no_more_diagnoses = set()
            no_more_medications = set()
            no_more_procedures = set()
            no_more_dose = set()

        # Informations per visit
        # Create dictionaries to store information per visit
        meds_per_visit = {}
        diag_per_visit = {}
        proc_per_visit = {}
        dose_per_visit = {}

        for visit_idx in range(landmark_idx + 1):
            # Get the visit information
            visit = group.iloc[visit_idx]
            # Split the lists into individual items and store them
            # in the dictionaries, Handling NaN values
            meds  = visit['medication_list'].split('\n') if pd.notna(visit['medication_list']) else []
            diags = visit['diagnosis_list'].split('\n')  if pd.notna(visit['diagnosis_list'])  else []
            procs = visit['procedure_list'].split('\n')  if pd.notna(visit['procedure_list'])  else []
            dose  = visit['dose_list'].split('\n')       if pd.notna(visit['dose_list'])       else []
            
            # Store the lists in the dictionaries
            meds_per_visit[visit_idx + 1] = meds
            diag_per_visit[visit_idx + 1] = diags
            proc_per_visit[visit_idx + 1] = procs
            dose_per_visit[visit_idx + 1] = dose

        # Create text representations of the lists
        # Needed for naive and no_narratives prompts
        unique_vals = set(all_visit_so_far['medication_list'])
        if len(unique_vals) == 1 and (pd.isna(next(iter(unique_vals))) or next(iter(unique_vals)) == ''):
            meds_text = ''
        else:
            meds_text = '\n'.join(all_visit_so_far['medication_list'])

        unique_vals = set(all_visit_so_far['diagnosis_list'])
        if len(unique_vals) == 1 and (pd.isna(next(iter(unique_vals))) or next(iter(unique_vals)) == ''):
            diag_text = ''
        else:
            diag_text = '\n'.join(all_visit_so_far['diagnosis_list'])

        unique_vals = set(all_visit_so_far['procedure_list'])
        if len(unique_vals) == 1 and (pd.isna(next(iter(unique_vals))) or next(iter(unique_vals)) == ''):
            proc_text = ''
        else:
            proc_text = '\n'.join(all_visit_so_far['procedure_list'])
        
        unique_vals = set(all_visit_so_far['dose_list'])
        if len(unique_vals) == 1 and (pd.isna(next(iter(unique_vals))) or next(iter(unique_vals)) == ''):
            dose_text = ''
        else:
            dose_text = '\n'.join(all_visit_so_far['dose_list'])

        days_to_death = current_visit['days_from_last_visit_to_death']
        death_in_90days = 1 if (not np.isnan(days_to_death) and days_to_death <= 90) else 0

        landmark_rows.append({
            'subject_id': current_visit['subject_id'],
            'hadm_id': current_visit['hadm_id'],
            'admission_category': current_visit['admission_category'],
            'landmark_visit': landmark_idx + 1,
            'age_at_landmark': current_visit['age_at_event'],
            'gender': current_visit['gender'],
            'num_total_visits': n_visits,  # No used
            'days_from_last_visit': current_visit['days_until_next_visit'],
            'death_in_90days': death_in_90days,

            'med_text': meds_text,   # naive no_narratives
            'diag_text': diag_text,  # naive no_narratives
            'proc_text': proc_text,  # naive no_narratives
            'dose_text': dose_text,  # naive no_narratives

            'new_medications': new_medications,         # XGBoost
            'new_diagnoses': new_diagnoses,             # XGBoost
            'new_procedures': new_procedures,           # XGBoost
            'new_dose': new_dose,                       # XGBoost
            'no_more_diagnoses': no_more_diagnoses,     # XGBoost
            'no_more_medications': no_more_medications, # XGBoost
            'no_more_procedures': no_more_procedures,   # XGBoost
            'no_more_dose': no_more_dose,               # XGBoost
 
            'meds_per_visit': meds_per_visit,           # full_* and compact_*
            'diag_per_visit': diag_per_visit,           # full_* and compact_*
            'proc_per_visit': proc_per_visit,           # full_* and compact_* 
            'dose_per_visit': dose_per_visit            # full_* and compact_*
        })

landmark_df = pd.DataFrame(landmark_rows)

In [19]:
len(patients_2_check) == 0

True

In [20]:
landmark_df['subject_id'].nunique()

223452

In [21]:
landmark_df[landmark_df['subject_id']==11530780]

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,days_from_last_visit,death_in_90days,med_text,...,new_procedures,new_dose,no_more_diagnoses,no_more_medications,no_more_procedures,no_more_dose,meds_per_visit,diag_per_visit,proc_per_visit,dose_per_visit
82093,11530780,26728216,NORMAL,1,22,F,3,-1.0,0,Albuterol Inhaler\nPrenatal Vitamins\nAcetamin...,...,{},{1-2 PUFF of Albuterol Inhaler in INH through ...,{},{},{},{},"{1: ['Albuterol Inhaler', 'Prenatal Vitamins',...",{1: ['Other current conditions classifiable el...,{1: ['']},{1: ['1-2 PUFF of Albuterol Inhaler in INH thr...
82094,11530780,20881796,NORMAL,2,22,F,3,80.0,0,Albuterol Inhaler\nPrenatal Vitamins\nAcetamin...,...,{},{},{Cough},"{Prenatal Vitamins, Acetaminophen, Albuterol I...",{},{1-2 PUFF of Albuterol Inhaler in INH through ...,"{1: ['Albuterol Inhaler', 'Prenatal Vitamins',...",{1: ['Other current conditions classifiable el...,"{1: [''], 2: ['']}",{1: ['1-2 PUFF of Albuterol Inhaler in INH thr...
82095,11530780,29701295,EMERGENCY,3,22,F,3,-1.0,0,Albuterol Inhaler\nPrenatal Vitamins\nAcetamin...,...,"{Medical induction of labor, Insertion of intr...",{30 mL of Milk of Magnesia in UDCUP through PO...,{Other current conditions classifiable elsewhe...,{},{},{},"{1: ['Albuterol Inhaler', 'Prenatal Vitamins',...",{1: ['Other current conditions classifiable el...,"{1: [''], 2: [''], 3: ['Medical induction of l...",{1: ['1-2 PUFF of Albuterol Inhaler in INH thr...


In [ ]:
# Convert sets in specified columns to sorted newline-separated strings, leave other types unchanged
for col in ['new_medications', 'new_diagnoses', 'new_procedures', 'new_dose', 'no_more_diagnoses', 'no_more_medications', 'no_more_procedures', 'no_more_dose']:
    landmark_df[col] = landmark_df[col].apply(lambda x: '\n'.join(sorted(x)) if isinstance(x, set) else x)


In [23]:
landmark_df['subject_id'].nunique()

223452

In [24]:
landmark_df.to_csv("landmark_df_evo_test.csv", index=False, na_rep='')

In [25]:
landmark_df['diag_per_visit'][2]

{1: ['Portal hypertension',
  'Other ascites',
  'Cirrhosis of liver without mention of alcohol',
  'Unspecified viral hepatitis C without hepatic coma',
  'Chronic airway obstruction, not elsewhere classified',
  'Bipolar disorder, unspecified',
  'Posttraumatic stress disorder',
  'Personal history of tobacco use'],
 2: ['Unspecified viral hepatitis C with hepatic coma',
  'Other ascites',
  'Thrombocytopenia, unspecified',
  'Hyposmolality and/or hyponatremia',
  'Chronic airway obstruction, not elsewhere classified',
  'Cirrhosis of liver without mention of alcohol',
  'Asymptomatic human immunodeficiency virus [HIV] infection status',
  'Tobacco use disorder'],
 3: ['Other iatrogenic hypotension',
  'Chronic hepatitis C with hepatic coma',
  'Cachexia',
  'Hyposmolality and/or hyponatremia',
  'Other ascites',
  'Hyperpotassemia',
  'Tobacco use disorder',
  'Asymptomatic human immunodeficiency virus [HIV] infection status',
  'Do not resuscitate status',
  'Other dependence on ma

In [26]:
landmark_df.head()

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,days_from_last_visit,death_in_90days,med_text,...,new_procedures,new_dose,no_more_diagnoses,no_more_medications,no_more_procedures,no_more_dose,meds_per_visit,diag_per_visit,proc_per_visit,dose_per_visit
0,10000032,22595853,EMERGENCY,1,52,F,4,50.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,Percutaneous abdominal drainage,0.5 mL of Influenza Vaccine Quadrivalent in SY...,,,,,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...",{1: ['Percutaneous abdominal drainage']},{1: ['40 mg of Furosemide in TAB through PO/NG...
1,10000032,22841357,EMERGENCY,2,52,F,4,25.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,,1 CAP of Tiotropium Bromide in CAP through IH\...,"Bipolar disorder, unspecified\nPersonal histor...",Ipratropium Bromide Neb\nNicotine Patch\nPotas...,,1 NEB of Ipratropium Bromide Neb in VIAL throu...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...
2,10000032,29079034,EMERGENCY,3,52,F,4,11.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,,1 NEB of Albuterol 0.083% Neb Soln in VIAL thr...,"Thrombocytopenia, unspecified\nUnspecified vir...",Acetaminophen\nFurosemide\nSulfameth/Trimethop...,Percutaneous abdominal drainage,1 TAB of Sulfameth/Trimethoprim DS in TAB thro...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...
3,10000032,25742920,EMERGENCY,4,52,F,4,-1.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,Percutaneous abdominal drainage,1 BAG of 0.9% Sodium Chloride in BAG through I...,"Bipolar disorder, unspecified\nCachexia\nChron...",Albuterol 0.083% Neb Soln\nBisacodyl\nDocusate...,,1 NEB of Albuterol 0.083% Neb Soln in VIAL thr...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...
4,10000068,25022803,NORMAL,1,19,F,1,-1.0,0,,...,Other nonoperative respiratory measurements,,,,,,{1: ['']},"{1: ['Alcohol abuse, unspecified']}",{1: ['Other nonoperative respiratory measureme...,{1: ['']}


In [1]:
import pandas as pd
landmark_df_evo = pd.read_csv("landmark_df_evo_test.csv", na_values=['', 'None', 'NaN', 'na', 'nan'])
landmark_df_evo = landmark_df_evo.fillna('')
landmark_df_evo.head()

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,days_from_last_visit,death_in_90days,med_text,...,new_procedures,new_dose,no_more_diagnoses,no_more_medications,no_more_procedures,no_more_dose,meds_per_visit,diag_per_visit,proc_per_visit,dose_per_visit
0,10000032,22595853,EMERGENCY,1,52,F,4,50.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,Percutaneous abdominal drainage,0.5 mL of Influenza Vaccine Quadrivalent in SY...,,,,,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...",{1: ['Percutaneous abdominal drainage']},{1: ['40 mg of Furosemide in TAB through PO/NG...
1,10000032,22841357,EMERGENCY,2,52,F,4,25.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,,1 CAP of Tiotropium Bromide in CAP through IH\...,"Bipolar disorder, unspecified\nPersonal histor...",Ipratropium Bromide Neb\nNicotine Patch\nPotas...,,1 NEB of Ipratropium Bromide Neb in VIAL throu...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...
2,10000032,29079034,EMERGENCY,3,52,F,4,11.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,,1 NEB of Albuterol 0.083% Neb Soln in VIAL thr...,"Thrombocytopenia, unspecified\nUnspecified vir...",Acetaminophen\nFurosemide\nSulfameth/Trimethop...,Percutaneous abdominal drainage,1 TAB of Sulfameth/Trimethoprim DS in TAB thro...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...
3,10000032,25742920,EMERGENCY,4,52,F,4,-1.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,Percutaneous abdominal drainage,1 BAG of 0.9% Sodium Chloride in BAG through I...,"Bipolar disorder, unspecified\nCachexia\nChron...",Albuterol 0.083% Neb Soln\nBisacodyl\nDocusate...,,1 NEB of Albuterol 0.083% Neb Soln in VIAL thr...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...
4,10000068,25022803,NORMAL,1,19,F,1,-1.0,0,,...,Other nonoperative respiratory measurements,,,,,,{1: ['']},"{1: ['Alcohol abuse, unspecified']}",{1: ['Other nonoperative respiratory measureme...,{1: ['']}


In [6]:
landmark_df_evo['subject_id'].nunique()

223452

In [4]:
landmark_df_evo_original = pd.read_csv("landmark_df_evo.csv", na_values=['', 'None', 'NaN', 'na', 'nan'])

In [5]:
landmark_df_evo_original['subject_id'].nunique()

168415

In [4]:
import ast
ast.literal_eval(landmark_df_evo['meds_per_visit'][0])

{1: ['Furosemide',
  'Ipratropium Bromide Neb',
  'Potassium Chloride',
  'Sodium Chloride 0.9%  Flush',
  'Raltegravir',
  'Spironolactone',
  'Acetaminophen',
  'Influenza Vaccine Quadrivalent',
  'Albuterol Inhaler',
  'Emtricitabine-Tenofovir (Truvada)',
  'Heparin',
  'Nicotine Patch']}

In [5]:
# landmark_df.to_csv("landmark_df_new.csv", index=False, na_rep='Unknown')
# landmark_df_mew = pd.read_csv("landmark_df_new.csv")
# # Check if the two DataFrames are equal
# if landmark_df.equals(landmark_df_mew):
#     print("The DataFrames are equal.")
# else:
#     print("The DataFrames are not equal.")
#     # Optionally, you can print the differences
#     print("Differences:")
#     print(landmark_df.compare(landmark_df_mew))
    
# # landmark_df.head()
# landmark_df_mew['no_more_medications'][2]

In [ ]:
# Current in production now
# landmark_rows = []
# for patient_id, group in df.groupby('subject_id'):
#     group = group.reset_index(drop=True)
#     n_visits = group.shape[0]

#     for landmark_idx in range(n_visits):
#         current_visit = group.iloc[landmark_idx]
#         past_visits = group.iloc[:landmark_idx + 1]

#         # Here: to be more efficient - how to say confirmed/new? For now okay here

#         unique_vals = set(past_visits['medication_list'])
#         if len(unique_vals) == 1 and pd.isna(next(iter(unique_vals))):
#             meds_text = ''
#         else:
#             meds_text = ' '.join(past_visits['medication_list'].apply(safe_join))
        
#         unique_vals = set(past_visits['diagnosis_list'])
#         if len(unique_vals) == 1 and pd.isna(next(iter(unique_vals))):
#             diag_text = ''
#         else:
#             diag_text = ' '.join(past_visits['diagnosis_list'].apply(safe_join))
        
#         unique_vals = set(past_visits['procedure_list'])
#         if len(unique_vals) == 1 and pd.isna(next(iter(unique_vals))):
#             proc_text = ''
#         else:
#             proc_text = ' '.join(past_visits['procedure_list'].apply(safe_join))

#         # Define outcome
#         days_to_death = current_visit['days_from_last_visit_to_death']
#         death_in_90days = 1 if (not np.isnan(days_to_death) and days_to_death <= 90) else 0

#         landmark_rows.append({
#             'subject_id': current_visit['subject_id'],
#             'landmark_visit': landmark_idx + 1,
#             'age_at_landmark': current_visit['age_at_event'],
#             'gender': current_visit['gender'],
#             'num_total_visits': landmark_idx + 1,
#             'days_from_last_visit': current_visit['days_until_next_visit'],
#             'death_in_90days': death_in_90days,
#             'med_text': meds_text,
#             'diag_text': diag_text,
#             'proc_text': proc_text
#         })

# landmark_df_old = pd.DataFrame(landmark_rows)

In [7]:
# Convert gender explicitly into binary numeric values
landmark_df_evo['gender_numeric'] = landmark_df_evo['gender'].map({'F': 0, 'M': 1})
landmark_df_evo.head(5)

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,days_from_last_visit,death_in_90days,med_text,...,new_dose,no_more_diagnoses,no_more_medications,no_more_procedures,no_more_dose,meds_per_visit,diag_per_visit,proc_per_visit,dose_per_visit,gender_numeric
0,10000032,22595853,EMERGENCY,1,52,F,4,50.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,0.5 mL of Influenza Vaccine Quadrivalent in SY...,,,,,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...",{1: ['Percutaneous abdominal drainage']},{1: ['40 mg of Furosemide in TAB through PO/NG...,0
1,10000032,22841357,EMERGENCY,2,52,F,4,25.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,1 CAP of Tiotropium Bromide in CAP through IH\...,"Bipolar disorder, unspecified\nPersonal histor...",Ipratropium Bromide Neb\nNicotine Patch\nPotas...,,1 NEB of Ipratropium Bromide Neb in VIAL throu...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0
2,10000032,29079034,EMERGENCY,3,52,F,4,11.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,1 NEB of Albuterol 0.083% Neb Soln in VIAL thr...,"Thrombocytopenia, unspecified\nUnspecified vir...",Acetaminophen\nFurosemide\nSulfameth/Trimethop...,Percutaneous abdominal drainage,1 TAB of Sulfameth/Trimethoprim DS in TAB thro...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0
3,10000032,25742920,EMERGENCY,4,52,F,4,-1.0,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,...,1 BAG of 0.9% Sodium Chloride in BAG through I...,"Bipolar disorder, unspecified\nCachexia\nChron...",Albuterol 0.083% Neb Soln\nBisacodyl\nDocusate...,,1 NEB of Albuterol 0.083% Neb Soln in VIAL thr...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0
4,10000068,25022803,NORMAL,1,19,F,1,-1.0,0,,...,,,,,,{1: ['']},"{1: ['Alcohol abuse, unspecified']}",{1: ['Other nonoperative respiratory measureme...,{1: ['']},0


In [8]:
landmark_df = landmark_df_evo.copy()
# landmark_df.to_csv('landmark_df_2.csv', index=False, na_rep = 'Unknown')

Still not equals. Something is missing...

In [9]:
# Ensure your DataFrame is sorted by patient and landmark_visit
landmark_df = landmark_df.sort_values(['subject_id', 'landmark_visit']).reset_index(drop=True)

# Function to perform transformation
def transform_to_past_variable(group):
    group = group.sort_values('landmark_visit').copy()
    # Shift the values down (future to past)
    group['days_since_last_visit'] = group['days_from_last_visit'].shift(1)
    # First landmark always has -1
    group['days_since_last_visit'].iloc[0] = -1
    return group

# Apply the transformation per patient
landmark_df = landmark_df.groupby('subject_id').apply(transform_to_past_variable).reset_index(drop=True)

# Drop the original future-looking column (optional but strongly recommended)
landmark_df.drop(columns=['days_from_last_visit'], inplace=True)

# Check the result clearly
print(landmark_df[['subject_id', 'landmark_visit', 'days_since_last_visit']].head(15))

    subject_id  landmark_visit  days_since_last_visit
0     10000032               1                   -1.0
1     10000032               2                   50.0
2     10000032               3                   25.0
3     10000032               4                   11.0
4     10000068               1                   -1.0
5     10000084               1                   -1.0
6     10000084               2                   32.0
7     10000108               1                   -1.0
8     10000117               1                   -1.0
9     10000117               2                  672.0
10    10000161               1                   -1.0
11    10000248               1                   -1.0
12    10000280               1                   -1.0
13    10000560               1                   -1.0
14    10000635               1                   -1.0


In [10]:
landmark_df = landmark_df.rename(columns={'days_since_last_visit':'days_since_previous_visit'})
landmark_df.head(10)

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,...,no_more_diagnoses,no_more_medications,no_more_procedures,no_more_dose,meds_per_visit,diag_per_visit,proc_per_visit,dose_per_visit,gender_numeric,days_since_previous_visit
0,10000032,22595853,EMERGENCY,1,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,,,,,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...",{1: ['Percutaneous abdominal drainage']},{1: ['40 mg of Furosemide in TAB through PO/NG...,0,-1.0
1,10000032,22841357,EMERGENCY,2,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"Bipolar disorder, unspecified\nPersonal histor...",Ipratropium Bromide Neb\nNicotine Patch\nPotas...,,1 NEB of Ipratropium Bromide Neb in VIAL throu...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,50.0
2,10000032,29079034,EMERGENCY,3,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"Thrombocytopenia, unspecified\nUnspecified vir...",Acetaminophen\nFurosemide\nSulfameth/Trimethop...,Percutaneous abdominal drainage,1 TAB of Sulfameth/Trimethoprim DS in TAB thro...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,25.0
3,10000032,25742920,EMERGENCY,4,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"Bipolar disorder, unspecified\nCachexia\nChron...",Albuterol 0.083% Neb Soln\nBisacodyl\nDocusate...,,1 NEB of Albuterol 0.083% Neb Soln in VIAL thr...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,11.0
4,10000068,25022803,NORMAL,1,19,F,1,0,,"Alcohol abuse, unspecified",...,,,,,{1: ['']},"{1: ['Alcohol abuse, unspecified']}",{1: ['Other nonoperative respiratory measureme...,{1: ['']},0,-1.0
5,10000084,23052089,EMERGENCY,1,72,M,2,1,Pramipexole\nPravastatin\nrivastigmine\nSenna\...,Neurocognitive disorder with Lewy bodies\nDeme...,...,,,,,"{1: ['Pramipexole', 'Pravastatin', 'rivastigmi...",{1: ['Neurocognitive disorder with Lewy bodies...,{1: ['']},{1: ['0.5 mg of Pramipexole in TAB through PO/...,1,-1.0
6,10000084,29888819,NORMAL,2,72,M,2,1,Pramipexole\nPravastatin\nrivastigmine\nSenna\...,Neurocognitive disorder with Lewy bodies\nDeme...,...,Neurocognitive disorder with Lewy bodies\nRepe...,Heparin\nPolyethylene Glycol\nPramipexole\nPra...,,0.125 mg of Pramipexole in TAB through PO/NG\n...,"{1: ['Pramipexole', 'Pravastatin', 'rivastigmi...",{1: ['Neurocognitive disorder with Lewy bodies...,"{1: [''], 2: ['']}",{1: ['0.5 mg of Pramipexole in TAB through PO/...,1,32.0
7,10000108,27250926,NORMAL,1,25,M,1,0,,Cellulitis and abscess of oral soft tissues\nO...,...,,,,,{1: ['']},{1: ['Cellulitis and abscess of oral soft tiss...,{1: ['']},{1: ['']},1,-1.0
8,10000117,22927623,NORMAL,1,55,F,2,0,Heparin\nSodium Chloride 0.9% Flush,"Dysphagia, unspecified\nOther specified sympto...",...,,,,,"{1: ['Heparin', 'Sodium Chloride 0.9% Flush']}","{1: ['Dysphagia, unspecified', 'Other specifie...",{1: ['']},"{1: ['5000 UNIT of Heparin in mL through SC', ...",0,-1.0
9,10000117,27988844,NORMAL,2,57,F,2,0,Heparin\nSodium Chloride 0.9% Flush\nIso-Osmo...,"Dysphagia, unspecified\nOther specified sympto...",...,Angiodysplasia of stomach and duodenum without...,Heparin,,3 mL of Sodium Chloride 0.9% Flush in SYR thr...,"{1: ['Heparin', 'Sodium Chloride 0.9% Flush']...","{1: ['Dy

In [11]:
landmark_df['subject_id'].nunique()

223452

In [ ]:
landmark_df.to_csv('landmark_df_evo_test.csv', index=False, na_rep = '')

In [2]:
landmark_df_evo = pd.read_csv("landmark_df_evo_test.csv", na_values=['', 'None', 'NaN', 'na', 'nan'])
landmark_df_evo = landmark_df_evo.fillna('')
landmark_df_evo.head()

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,...,no_more_diagnoses,no_more_medications,no_more_procedures,no_more_dose,meds_per_visit,diag_per_visit,proc_per_visit,dose_per_visit,gender_numeric,days_since_previous_visit
0,10000032,22595853,EMERGENCY,1,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,,,,,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...",{1: ['Percutaneous abdominal drainage']},{1: ['40 mg of Furosemide in TAB through PO/NG...,0,-1.0
1,10000032,22841357,EMERGENCY,2,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"Bipolar disorder, unspecified\nPersonal histor...",Ipratropium Bromide Neb\nNicotine Patch\nPotas...,,1 NEB of Ipratropium Bromide Neb in VIAL throu...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,50.0
2,10000032,29079034,EMERGENCY,3,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"Thrombocytopenia, unspecified\nUnspecified vir...",Acetaminophen\nFurosemide\nSulfameth/Trimethop...,Percutaneous abdominal drainage,1 TAB of Sulfameth/Trimethoprim DS in TAB thro...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,25.0
3,10000032,25742920,EMERGENCY,4,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"Bipolar disorder, unspecified\nCachexia\nChron...",Albuterol 0.083% Neb Soln\nBisacodyl\nDocusate...,,1 NEB of Albuterol 0.083% Neb Soln in VIAL thr...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,11.0
4,10000068,25022803,NORMAL,1,19,F,1,0,,"Alcohol abuse, unspecified",...,,,,,{1: ['']},"{1: ['Alcohol abuse, unspecified']}",{1: ['Other nonoperative respiratory measureme...,{1: ['']},0,-1.0


In [3]:
landmark_df_evo['subject_id'].nunique()

223452

In [5]:
landmark_df_evo.to_csv('landmark_df_evo_correct.csv', index=False, na_rep = '')

In [ ]:
# landmark_df_2.dtypes
# landmark_df_2.proc_text[5]

nan

In [ ]:
# landmark_df_1.dtypes
# landmark_df_1.proc_text[5]

'Unknown'

In [ ]:
# landmark_df_2.to_csv('landmark_df_2.csv', index=False, na_rep = 'Unknown')

In [ ]:
# landmark_df_3 = pd.read_csv('landmark_df_2.csv')

In [16]:
# landmark_df_3.head(20)

In [17]:
# landmark_df_3.equals(landmark_df_1)

Okay, so the code here it's the one that has been generated trough xgboost.ipynb. The Unknown have been creating saving two times the dataframe. First, creating NaN and then creating Unknown.

#### Why not all blank spaces are 'Unknown'?
Let's take the 10000084 subject. In the second landmark visit we don't have anymore Unknown, but, rather, ''. We need to investigate further.
Solved! See the code above, has already the solution.

## Exploring the narrative_prompt

In [1]:
import pandas as pd
import numpy as np
import ast

In [2]:
landmark_df_evo = pd.read_csv("landmark_df_evo_correct.csv", na_values=['', 'None', 'NaN', 'na', 'nan'])
landmark_df_evo = landmark_df_evo.fillna('')

In [3]:
landmark_df_evo.head(10)

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,...,no_more_diagnoses,no_more_medications,no_more_procedures,no_more_dose,meds_per_visit,diag_per_visit,proc_per_visit,dose_per_visit,gender_numeric,days_since_previous_visit
0,10000032,22595853,EMERGENCY,1,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,,,,,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...",{1: ['Percutaneous abdominal drainage']},{1: ['40 mg of Furosemide in TAB through PO/NG...,0,-1.0
1,10000032,22841357,EMERGENCY,2,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"Bipolar disorder, unspecified\nPersonal histor...",Ipratropium Bromide Neb\nNicotine Patch\nPotas...,,1 NEB of Ipratropium Bromide Neb in VIAL throu...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,50.0
2,10000032,29079034,EMERGENCY,3,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"Thrombocytopenia, unspecified\nUnspecified vir...",Acetaminophen\nFurosemide\nSulfameth/Trimethop...,Percutaneous abdominal drainage,1 TAB of Sulfameth/Trimethoprim DS in TAB thro...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,25.0
3,10000032,25742920,EMERGENCY,4,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"Bipolar disorder, unspecified\nCachexia\nChron...",Albuterol 0.083% Neb Soln\nBisacodyl\nDocusate...,,1 NEB of Albuterol 0.083% Neb Soln in VIAL thr...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,11.0
4,10000068,25022803,NORMAL,1,19,F,1,0,,"Alcohol abuse, unspecified",...,,,,,{1: ['']},"{1: ['Alcohol abuse, unspecified']}",{1: ['Other nonoperative respiratory measureme...,{1: ['']},0,-1.0
5,10000084,23052089,EMERGENCY,1,72,M,2,1,Pramipexole\nPravastatin\nrivastigmine\nSenna\...,Neurocognitive disorder with Lewy bodies\nDeme...,...,,,,,"{1: ['Pramipexole', 'Pravastatin', 'rivastigmi...",{1: ['Neurocognitive disorder with Lewy bodies...,{1: ['']},{1: ['0.5 mg of Pramipexole in TAB through PO/...,1,-1.0
6,10000084,29888819,NORMAL,2,72,M,2,1,Pramipexole\nPravastatin\nrivastigmine\nSenna\...,Neurocognitive disorder with Lewy bodies\nDeme...,...,Neurocognitive disorder with Lewy bodies\nRepe...,Heparin\nPolyethylene Glycol\nPramipexole\nPra...,,0.125 mg of Pramipexole in TAB through PO/NG\n...,"{1: ['Pramipexole', 'Pravastatin', 'rivastigmi...",{1: ['Neurocognitive disorder with Lewy bodies...,"{1: [''], 2: ['']}",{1: ['0.5 mg of Pramipexole in TAB through PO/...,1,32.0
7,10000108,27250926,NORMAL,1,25,M,1,0,,Cellulitis and abscess of oral soft tissues\nO...,...,,,,,{1: ['']},{1: ['Cellulitis and abscess of oral soft tiss...,{1: ['']},{1: ['']},1,-1.0
8,10000117,22927623,NORMAL,1,55,F,2,0,Heparin\nSodium Chloride 0.9% Flush,"Dysphagia, unspecified\nOther specified sympto...",...,,,,,"{1: ['Heparin', 'Sodium Chloride 0.9% Flush']}","{1: ['Dysphagia, unspecified', 'Other specifie...",{1: ['']},"{1: ['5000 UNIT of Heparin in mL through SC', ...",0,-1.0
9,10000117,27988844,NORMAL,2,57,F,2,0,Heparin\nSodium Chloride 0.9% Flush\nIso-Osmo...,"Dysphagia, unspecified\nOther specified sympto...",...,Angiodysplasia of stomach and duodenum without...,Heparin,,3 mL of Sodium Chloride 0.9% Flush in SYR thr...,"{1: ['Heparin', 'Sodium Chloride 0.9% Flush']...","{1: ['Dy

In [4]:
landmark_df_evo['med_text'][1].split('\n')


['Furosemide',
 'Ipratropium Bromide Neb',
 'Potassium Chloride',
 'Sodium Chloride 0.9%  Flush',
 'Raltegravir',
 'Spironolactone',
 'Acetaminophen',
 'Influenza Vaccine Quadrivalent',
 'Albuterol Inhaler',
 'Emtricitabine-Tenofovir (Truvada)',
 'Heparin',
 'Nicotine Patch',
 'Furosemide',
 'Rifaximin',
 'Sodium Chloride 0.9%  Flush',
 'Albuterol Inhaler',
 'Calcium Carbonate',
 'Acetaminophen',
 'Influenza Vaccine Quadrivalent',
 'Albumin 25% (12.5g / 50mL)',
 'Tiotropium Bromide',
 'Emtricitabine-Tenofovir (Truvada)',
 'Lactulose',
 'Raltegravir',
 'Heparin',
 'Sulfameth/Trimethoprim DS']

In [5]:
ast.literal_eval(landmark_df_evo['dose_per_visit'][2])

{1: ['40 mg of Furosemide in TAB through PO/NG',
  '1 NEB of Ipratropium Bromide Neb in VIAL through IH',
  '20 mg of Furosemide in TAB through PO/NG',
  '40 mEq of Potassium Chloride in TAB through PO',
  '3 mL of Sodium Chloride 0.9%  Flush in SYR through IV',
  '400 mg of Raltegravir in TAB through PO',
  '50 mg of Spironolactone in TAB through PO/NG',
  '500 mg of Acetaminophen in TAB through PO/NG',
  '0.5 mL of Influenza Vaccine Quadrivalent in SYR through IM',
  '2 PUFF of Albuterol Inhaler in INH through IH',
  '1 TAB of Emtricitabine-Tenofovir (Truvada) in TAB through PO',
  '50 mg of Spironolactone in TAB through PO/NG',
  '5000 UNIT of Heparin in mL through SC',
  '14 mg of Nicotine Patch in PTCH through TD'],
 2: ['40 mg of Furosemide in TAB through PO/NG',
  '550 mg of Rifaximin in TAB through PO/NG',
  '3 mL of Sodium Chloride 0.9%  Flush in SYR through IV',
  '2 PUFF of Albuterol Inhaler in INH through IH',
  '1250 mg of Calcium Carbonate in TAB through PO/NG',
  '20 mg 

In [6]:
# PEr test e debug
row = landmark_df_evo.iloc[3]

In [7]:
ast.literal_eval(row['meds_per_visit'])[1]

['Furosemide',
 'Ipratropium Bromide Neb',
 'Potassium Chloride',
 'Sodium Chloride 0.9%  Flush',
 'Raltegravir',
 'Spironolactone',
 'Acetaminophen',
 'Influenza Vaccine Quadrivalent',
 'Albuterol Inhaler',
 'Emtricitabine-Tenofovir (Truvada)',
 'Heparin',
 'Nicotine Patch']

In [8]:
narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days for {row['age_at_landmark']}-year-old {row['gender']} patient\n"
current_visit = row['landmark_visit']
narrative += f"Visit number {current_visit}\n"

if row['days_since_previous_visit'] != -1:
    narrative += f"Last visit happened {row['days_since_previous_visit']} days ago."

narrative += "\nDiagnosis history:"
for past_visit, diags in ast.literal_eval(row['diag_per_visit']).items():
    narrative += f"\nIn visit {past_visit}: {', '.join(diags)}."

narrative += "\nPrescriptions history:"
for past_visit, meds in ast.literal_eval(row['meds_per_visit']).items():
        narrative += f"\nIn visit {past_visit}: {', '.join(meds)}."

narrative += "\nProcedures history:"
for past_visit, proc in ast.literal_eval(row['proc_per_visit']).items():
        narrative += f"\nIn visit {past_visit}: {', '.join(proc)}."



# if pd.notna(row['diag_text']) and row['diag_text'].strip():
#     narrative += f" Medical history includes: {row['diag_text']}."
# if pd.notna(row['med_text']) and row['med_text'].strip():
#     narrative += f" Current medications are: {row['med_text']}."
# if pd.notna(row['proc_text']) and row['proc_text'].strip():
#     narrative += f" Procedures performed: {row['proc_text']}."

In [9]:
print(narrative)

You are a Doctor.
What is the probability of death in the next 90 days for 52-year-old F patient
Visit number 4
Last visit happened 11.0 days ago.
Diagnosis history:
In visit 1: Portal hypertension, Other ascites, Cirrhosis of liver without mention of alcohol, Unspecified viral hepatitis C without hepatic coma, Chronic airway obstruction, not elsewhere classified, Bipolar disorder, unspecified, Posttraumatic stress disorder, Personal history of tobacco use.
In visit 2: Unspecified viral hepatitis C with hepatic coma, Other ascites, Thrombocytopenia, unspecified, Hyposmolality and/or hyponatremia, Chronic airway obstruction, not elsewhere classified, Cirrhosis of liver without mention of alcohol, Asymptomatic human immunodeficiency virus [HIV] infection status, Tobacco use disorder.
In visit 3: Other iatrogenic hypotension, Chronic hepatitis C with hepatic coma, Cachexia, Hyposmolality and/or hyponatremia, Other ascites, Hyperpotassemia, Tobacco use disorder, Asymptomatic human immunode

Here we could obtain a more-time-series-like narrative, adding as well the temporal aspect on all visits. For doing so we cannot take just the row of the landmark_evo dataset, since there is just the temporal information of the last visit. We should do another step.

In [10]:
landmark_df_evo.columns

Index(['subject_id', 'hadm_id', 'admission_category', 'landmark_visit',
       'age_at_landmark', 'gender', 'num_total_visits', 'death_in_90days',
       'med_text', 'diag_text', 'proc_text', 'dose_text', 'new_medications',
       'new_diagnoses', 'new_procedures', 'new_dose', 'no_more_diagnoses',
       'no_more_medications', 'no_more_procedures', 'no_more_dose',
       'meds_per_visit', 'diag_per_visit', 'proc_per_visit', 'dose_per_visit',
       'gender_numeric', 'days_since_previous_visit'],
      dtype='object')

In [11]:
landmark_df_evo['days_since_previous_visit_cumulate'] = landmark_df_evo.groupby('subject_id')['days_since_previous_visit'].transform(lambda x: [list(x[:i+1]) for i in range(len(x))])

In [12]:
landmark_df_evo['days_since_previous_visit_cumulate_sum'] = landmark_df_evo['days_since_previous_visit_cumulate'].apply(lambda x: np.cumsum([y for y in x if y > 0][::-1])[::-1] if isinstance(x, list) else [-1])

In [13]:
landmark_df_evo.head(4)

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,...,no_more_procedures,no_more_dose,meds_per_visit,diag_per_visit,proc_per_visit,dose_per_visit,gender_numeric,days_since_previous_visit,days_since_previous_visit_cumulate,days_since_previous_visit_cumulate_sum
0,10000032,22595853,EMERGENCY,1,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,,,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...",{1: ['Percutaneous abdominal drainage']},{1: ['40 mg of Furosemide in TAB through PO/NG...,0,-1.0,[-1.0],[]
1,10000032,22841357,EMERGENCY,2,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,,1 NEB of Ipratropium Bromide Neb in VIAL throu...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,50.0,"[-1.0, 50.0]",[50.0]
2,10000032,29079034,EMERGENCY,3,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,Percutaneous abdominal drainage,1 TAB of Sulfameth/Trimethoprim DS in TAB thro...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,25.0,"[-1.0, 50.0, 25.0]","[75.0, 25.0]"
3,10000032,25742920,EMERGENCY,4,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,,1 NEB of Albuterol 0.083% Neb Soln in VIAL thr...,"{1: ['Furosemide', 'Ipratropium Bromide Neb', ...","{1: ['Portal hypertension', 'Other ascites', '...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,11.0,"[-1.0, 50.0, 25.0, 11.0]","[86.0, 36.0, 11.0]"


In [14]:

row = landmark_df_evo.iloc[3]
narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days from today for this {row['age_at_landmark']}-year-old {row['gender']} patient\n"
current_visit = row['landmark_visit']
narrative += f"Today is the {current_visit} visit.\n"

max_visit = int(row['landmark_visit'])

# if row['days_since_previous_visit'] != -1:
#     narrative += f"Last visit happened {row['days_since_previous_visit']} days ago."

narrative += "\nDiagnosis history:"
for past_visit, diags in reversed(list(ast.literal_eval(row['diag_per_visit']).items())):
    if past_visit == max_visit:
        narrative += f"\nToday: {', '.join(diags)}."
    else:
        narrative += f"\n{row['days_since_previous_visit_cumulate_sum'][int(past_visit) -1]}  days ago: {', '.join(diags)}."

narrative += "\nPrescriptions history:"
for past_visit, meds in reversed(ast.literal_eval(row['meds_per_visit']).items()):
    if past_visit ==  max_visit:
        narrative += f"\nToday: {', '.join(meds)}."
    else:
        narrative += f"\n{row['days_since_previous_visit_cumulate_sum'][past_visit-1]} days ago: {', '.join(meds)}."

narrative += "\nProcedures history:"
for past_visit, proc in reversed(ast.literal_eval(row['proc_per_visit']).items()):
    if past_visit == max_visit:
        narrative += f"\nToday: {', '.join(proc)}."
    else:
        narrative += f"\n{row['days_since_previous_visit_cumulate_sum'][past_visit-1]} days ago: {', '.join(proc)}."


In [15]:
print(narrative)

You are a Doctor.
What is the probability of death in the next 90 days from today for this 52-year-old F patient
Today is the 4 visit.

Diagnosis history:
Today: Chronic hepatitis C without mention of hepatic coma, Other ascites, Other dependence on machines, supplemental oxygen, Cirrhosis of liver without mention of alcohol, Hyperpotassemia, Hyposmolality and/or hyponatremia, Chronic airway obstruction, not elsewhere classified, Asymptomatic human immunodeficiency virus [HIV] infection status, Tobacco use disorder, Diarrhea.
11.0  days ago: Other iatrogenic hypotension, Chronic hepatitis C with hepatic coma, Cachexia, Hyposmolality and/or hyponatremia, Other ascites, Hyperpotassemia, Tobacco use disorder, Asymptomatic human immunodeficiency virus [HIV] infection status, Do not resuscitate status, Other dependence on machines, supplemental oxygen, Chronic airway obstruction, not elsewhere classified, Bipolar disorder, unspecified, Cirrhosis of liver without mention of alcohol.
36.0  da

In [16]:
def full_narrative(row):
    narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days from today for this {row['age_at_landmark']}-year-old {row['gender']} patient\n"
    current_visit = row['landmark_visit']
    narrative += f"Today is the {current_visit} visit.\n"

    max_visit = int(row['landmark_visit'])

    # if row['days_since_previous_visit'] != -1:
    #     narrative += f"Last visit happened {row['days_since_previous_visit']} days ago."

    narrative += "\nDiagnosis history:"
    for past_visit, diags in reversed(list(ast.literal_eval(row['diag_per_visit']).items())):
        if past_visit == max_visit:
            narrative += f"\nToday: {', '.join(diags)}."
        else:
            narrative += f"\n{int(row['days_since_previous_visit_cumulate_sum'][int(past_visit) -1])} days ago: {', '.join(diags)}."

    narrative += "\nPrescriptions history:"
    for past_visit, meds in reversed(ast.literal_eval(row['meds_per_visit']).items()):
        if past_visit ==  max_visit:
            narrative += f"\nToday: {', '.join(meds)}."
        else:
            narrative += f"\n{int(row['days_since_previous_visit_cumulate_sum'][past_visit-1])} days ago: {', '.join(meds)}."

    narrative += "\nProcedures history:"
    for past_visit, proc in reversed(ast.literal_eval(row['proc_per_visit']).items()):
        if past_visit == max_visit:
            narrative += f"\nToday: {', '.join(proc)}."
        else:
            narrative += f"\n{int(row['days_since_previous_visit_cumulate_sum'][past_visit-1])} days ago: {', '.join(proc)}."
    
    return narrative


In [17]:
print(full_narrative(landmark_df_evo.iloc[2]))

You are a Doctor.
What is the probability of death in the next 90 days from today for this 52-year-old F patient
Today is the 3 visit.

Diagnosis history:
Today: Other iatrogenic hypotension, Chronic hepatitis C with hepatic coma, Cachexia, Hyposmolality and/or hyponatremia, Other ascites, Hyperpotassemia, Tobacco use disorder, Asymptomatic human immunodeficiency virus [HIV] infection status, Do not resuscitate status, Other dependence on machines, supplemental oxygen, Chronic airway obstruction, not elsewhere classified, Bipolar disorder, unspecified, Cirrhosis of liver without mention of alcohol.
25 days ago: Unspecified viral hepatitis C with hepatic coma, Other ascites, Thrombocytopenia, unspecified, Hyposmolality and/or hyponatremia, Chronic airway obstruction, not elsewhere classified, Cirrhosis of liver without mention of alcohol, Asymptomatic human immunodeficiency virus [HIV] infection status, Tobacco use disorder.
75 days ago: Portal hypertension, Other ascites, Cirrhosis of 

In [18]:
import ast
from collections import Counter

def narrative_prompt_compact(row):
    narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days for {row['age_at_landmark']}-year-old {row['gender']} patient?\n"
    current_visit = row['landmark_visit']
    type = row['admission_category']
    narrative += f"Visit number {current_visit} - {type} \n"

    if row['days_since_previous_visit'] != -1:
        narrative += f"Last visit happened {row['days_since_previous_visit']} days ago."

    # --- Diagnosi ---
    narrative += "\nDIAGNOSIS HISTORY:"
    diag_per_visit = ast.literal_eval(row['diag_per_visit'])
    
    # Conta frequenze
    all_diags = []
    for diags in diag_per_visit.values():
        all_diags.extend(diags)
    diag_counts = Counter(all_diags)

    # Diagnosi croniche (almeno 2 visite)
    chronic_diags = [d for d, c in diag_counts.items() if c >= 2]

    # Diagnosi nuove solo in questa visita
    current_diags = diag_per_visit[int(current_visit)]
    new_diags = [d for d in current_diags if diag_counts[d] == 1]

    if chronic_diags:
        narrative += f"\nChronic diagnoses: {', '.join(chronic_diags)}."
    if new_diags:
        narrative += f"\nNew diagnoses in this visit: {', '.join(new_diags)}."
    if not chronic_diags and not new_diags:
        narrative += "\nNo diagnoses recorded."

    # --- Farmaci ---
    narrative += "\nPRESCRIPTIONS HISTORY:"
    meds_per_visit = ast.literal_eval(row['meds_per_visit'])
    
    all_meds = []
    for meds in meds_per_visit.values():
        all_meds.extend(meds)
    med_counts = Counter(all_meds)

    chronic_meds = [m for m, c in med_counts.items() if c >= 2]
    current_meds = meds_per_visit[int(current_visit)]
    new_meds = [m for m in current_meds if med_counts[m] == 1]

    if chronic_meds:
        narrative += f"\nChronic medications: {', '.join(chronic_meds)}."
    if new_meds:
        narrative += f"\nNew medications in this visit: {', '.join(new_meds)}."
    if not chronic_meds and not new_meds:
        narrative += "\nNo medications recorded."

    # --- Procedure ---
    narrative += "\nPROCEDURES HISTORY:"
    proc_per_visit = ast.literal_eval(row['proc_per_visit'])

    all_proc = []
    for procs in proc_per_visit.values():
        all_proc.extend(procs)
    proc_counts = Counter(all_proc)

    chronic_proc = [p for p, c in proc_counts.items() if c >= 2]
    current_proc = proc_per_visit[int(current_visit)]
    new_proc = [p for p in current_proc if proc_counts[p] == 1]

    if chronic_proc:
        narrative += f"\nChronic procedures: {', '.join(chronic_proc)}."
    if new_proc:
        narrative += f"\nNew procedures in this visit: {', '.join(new_proc)}."
    if not chronic_proc and not new_proc:
        narrative += "\nNo procedures recorded."

    return narrative

In [19]:
row = landmark_df_evo.iloc[3]
print(narrative_prompt_compact(row))

You are a Doctor.
What is the probability of death in the next 90 days for 52-year-old F patient?
Visit number 4 - EMERGENCY 
Last visit happened 11.0 days ago.
DIAGNOSIS HISTORY:
Chronic diagnoses: Other ascites, Cirrhosis of liver without mention of alcohol, Chronic airway obstruction, not elsewhere classified, Bipolar disorder, unspecified, Hyposmolality and/or hyponatremia, Asymptomatic human immunodeficiency virus [HIV] infection status, Tobacco use disorder, Hyperpotassemia, Other dependence on machines, supplemental oxygen.
New diagnoses in this visit: Chronic hepatitis C without mention of hepatic coma, Diarrhea.
PRESCRIPTIONS HISTORY:
Chronic medications: Furosemide, Sodium Chloride 0.9%  Flush, Raltegravir, Acetaminophen, Influenza Vaccine Quadrivalent, Albuterol Inhaler, Emtricitabine-Tenofovir (Truvada), Heparin, Rifaximin, Calcium Carbonate, Albumin 25% (12.5g / 50mL), Tiotropium Bromide, Lactulose, Fluticasone Propionate 110mcg, TraMADOL (Ultram).
New medications in this 